<a href="https://colab.research.google.com/github/fc63/gender-classification/blob/main/europarl_normalized/europarl_normalized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import warnings
import pandas as pd
import numpy as np
import os
import re
import pickle
import gc
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from google.colab import drive
from transformers import EarlyStoppingCallback

drive.mount('/content/drive')

# ignore warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# load dataset
dataset = load_dataset("samzirbo/europarl.en-es.gendered")

print("Libraries and datasets loaded!")

Mounted at /content/drive


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/144 [00:00<?, ?B/s]

europarl.en-es.simple.json:   0%|          | 0.00/526M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1419507 [00:00<?, ? examples/s]

Libraries and datasets loaded!


In [3]:
# convert to pandas
df = dataset['train'].to_pandas()

# remove neutral samples
df = df[df['gender'] != 'neutral'].copy()

# only keep English text
df = df[['en', 'gender']]

df.rename(columns={"en": "text"}, inplace=True)

df

,text,gender
0,in writing . - ( PT ) I would stress the poten...,male
2,"I would stress again that , unfortunately , th...",male
3,Of the EUR 500 million in funding made availab...,male
4,"I would , however , stress the fact that , fol...",male
5,on behalf of the PSE Group . - ( PL ) Mr Presi...,female
...,...,...
1419501,We witnessed both young and old expressing a v...,male
1419502,"This Parliament , other parliaments and politi...",male
1419503,The people have spoken on this issue and we mu...,male
1419505,It is an act of justice .,male


In [4]:
# normalization

df = df[~df["text"].str.contains(r'\([^()]*\)', regex=True)].copy()
df.loc[:, "text"] = df["text"].str.replace("-", "", regex=False)
df.loc[:, "text"] = df["text"].str.replace("–", "", regex=False)
df.loc[:, "text"] = df["text"].str.replace("\"", "'", regex=False)
df.loc[:, "text"] = df["text"].str.replace("%", "percent", regex=False)
df.loc[:, "text"] = df["text"].str.replace("’", "'", regex=False)
df.loc[:, "text"] = df["text"].str.replace(r"(?<!\s)\'(?!\s)", "", regex=True)
df.loc[:, "text"] = df["text"].str.replace(r"\s+\'\s+", "'", regex=True)
df.loc[:, "text"] = df["text"].str.replace(r"\s+\'(?=\S)", "'", regex=True)
df.loc[:, "text"] = df["text"].str.replace(r"(?<=\S)\'\s+", "'", regex=True)
df.loc[:, "text"] = df["text"].str.replace(r"(?<!\s)\'([^\']+?)\'(?=[.?!,:;])", r" '\1'", regex=True)
df.loc[:, "text"] = df["text"].str.replace(r"(?<!\s)\'([^\']+?)\'(?!\s)", r" '\1' ", regex=True)
df.loc[:, "text"] = df["text"].str.replace(r"i\.\s*e\.", "that is", regex=True)
df.loc[:, "text"] = df["text"].str.replace(r"e\.\s*g\.", "for example", regex=True)
df.loc[:, "text"] = df["text"].str.replace(r"etc\.", "and so on", regex=True)
df.loc[:, "text"] = df["text"].str.replace(r'(?<=\d)\s(?=\d)', '', regex=True)
df.loc[:, "text"] = df["text"].str.replace(r'(?<=\d)\.\s(?=\d)', '.', regex=True)
df.loc[:, "text"] = df["text"].str.replace(r"(?<!\s)\/(?!\s)", "", regex=True)
df.loc[:, "text"] = df["text"].str.replace(r"\s+\/\s+", "/", regex=True)
df.loc[:, "text"] = df["text"].str.replace(r"\s{2,}", " ", regex=True)
df.loc[:, "text"] = df["text"].str.strip()
df.loc[:, "text"] = df["text"].str.replace(r"(\.\s*){2,}", ".", regex=True)

def fix_punctuation_spacing(text):
    text = re.sub(r'\s+([,.!?;:])', r'\1', text)
    text = re.sub(r'([,.!?;:])(?=[^\s])', r'\1 ', text)
    return text.strip()

df.loc[:, 'text'] = df['text'].apply(fix_punctuation_spacing)
df = df[df["text"].str.len() >= 10].reset_index(drop=True)

df

,text,gender
0,"I would stress again that, unfortunately, the ...",male
1,Of the EUR 500 million in funding made availab...,male
2,"I would, however, stress the fact that, follow...",male
3,We must surely protect consumers'rights agains...,female
4,Thousands of companies throughout Europe are t...,female
...,...,...
1101250,We witnessed both young and old expressing a v...,male
1101251,"This Parliament, other parliaments and politic...",male
1101252,The people have spoken on this issue and we mu...,male
1101253,It is an act of justice.,male


In [6]:
os.makedirs('/content/drive/MyDrive/datasets', exist_ok=True)
with open('/content/drive/MyDrive/datasets/europarl_normalized.pkl', 'wb') as f:
    pickle.dump(df, f)